# Implementing Recommender Systems - Lab

## Introduction

In this lab, you'll practice creating a recommender system model using `surprise`. You'll also get the chance to create a more complete recommender system pipeline to obtain the top recommendations for a specific user.


## Objectives

In this lab you will: 

- Use surprise's built-in reader class to process data to work with recommender algorithms 
- Obtain a prediction for a specific user for a particular item 
- Introduce a new user with rating to a rating matrix and make recommendations for them 
- Create a function that will return the top n recommendations for a user 


For this lab, we will be using the famous 1M movie dataset. It contains a collection of user ratings for many different movies. In the last lesson, you were exposed to working with `surprise` datasets. In this lab, you will also go through the process of reading in a dataset into the `surprise` dataset format. To begin with, load the dataset into a Pandas DataFrame. Determine which columns are necessary for your recommendation system and drop any extraneous ones.

In [1]:
import pandas as pd
df = pd.read_csv('./ml-latest-small/ratings.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [2]:
# Drop unnecessary columns
new_df = df.drop('timestamp', axis=1)
new_df.head()

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


It's now time to transform the dataset into something compatible with `surprise`. In order to do this, you're going to need `Reader` and `Dataset` classes. There's a method in `Dataset` specifically for loading dataframes.

In [3]:
! pip install scikit-surprise

In [4]:
from surprise import Reader, Dataset
# read in values as Surprise dataset 
read = Reader(rating_scale=(0.5, 5))
data = Dataset.load_from_df(new_df, read)


Let's look at how many users and items we have in our dataset. If using neighborhood-based methods, this will help us determine whether or not we should perform user-user or item-item similarity

In [5]:
dataset = data.build_full_trainset()
print('Number of users: ', dataset.n_users, '\n')
print('Number of items: ', dataset.n_items)

Number of users:  610 

Number of items:  9724


## Determine the best model 

Now, compare the different models and see which ones perform best. For consistency sake, use RMSE to evaluate models. Remember to cross-validate! Can you get a model with a higher average RMSE on test data than 0.869?

In [6]:
# importing relevant libraries
from surprise.model_selection import cross_validate
from surprise.prediction_algorithms import SVD
from surprise.prediction_algorithms import KNNWithMeans, KNNBasic, KNNBaseline
from surprise.model_selection import GridSearchCV
import numpy as np

In [ ]:
## Perform a gridsearch with SVD
# ⏰ This cell may take several minutes to run
params = {
    'n_factors': [20, 40, 80, 100],
    'reg_all': [0.02, 0.01, 0.05, 0.1]
}

# Perform grid search
gs = GridSearchCV(
    SVD,
    params,
    measures=['rmse', 'mae'],
    cv=3
)

# Fit the grid search
gs.fit(data)

In [ ]:
# print out optimal parameters for SVD after GridSearch
# Print best RMSE and MAE scores
print(gs.best_score)

# Print best parameters
print(gs.best_params)

In [21]:
# cross validating with KNNBasic
# Instantiate model
knn_basic = KNNBasic()

# Cross validate
results_basic = cross_validate(
    knn_basic,
    data,
    measures=['RMSE', 'MAE'],
    cv=5,
    verbose=True
)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBasic on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9475  0.9465  0.9506  0.9525  0.9414  0.9477  0.0038  
MAE (testset)     0.7247  0.7261  0.7264  0.7304  0.7220  0.7259  0.0027  
Fit time          0.19    0.22    0.21    0.21    0.24    0.22    0.02    
Test time         1.66    1.72    1.72    1.75    1.69    1.71    0.03    


In [22]:
# print out the average RMSE score for the test set
print(np.mean(results_basic['test_rmse']))

0.9476877940121262


In [23]:
# cross validating with KNNBaseline
# Instantiate model
knn_baseline = KNNBaseline()

# Cross validate
results_baseline = cross_validate(
    knn_baseline,
    data,
    measures=['RMSE', 'MAE'],
    cv=5,
    verbose=True
)

Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBaseline on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8655  0.8731  0.8802  0.8732  0.8777  0.8739  0.0050  
MAE (testset)     0.6638  0.6678  0.6697  0.6676  0.6708  0.6679  0.0024  
Fit time          0.48    0.48    0.49    0.48    0.49    0.48    0.01    
Test time         2.28    2.23    2.40    2.42    2.77    2.42    0.19    


In [24]:
# print out the average score for the test set
# Average RMSE
print(np.mean(results_baseline['test_rmse']))

0.8739326533295744


Based off these outputs, it seems like the best performing model is the SVD model with `n_factors = 50` and a regularization rate of 0.05. Use that model or if you found one that performs better, feel free to use that to make some predictions.

## Making Recommendations

It's important that the output for the recommendation is interpretable to people. Rather than returning the `movie_id` values, it would be far more valuable to return the actual title of the movie. As a first step, let's read in the movies to a dataframe and take a peek at what information we have about them.

In [25]:
df_movies = pd.read_csv('./ml-latest-small/movies.csv')

In [26]:
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Making simple predictions
Just as a reminder, let's look at how you make a prediction for an individual user and item. First, we'll fit the SVD model we had from before.

In [27]:
svd = SVD(n_factors= 50, reg_all=0.05)
svd.fit(dataset)

In [28]:
svd.predict(2, 4)

Prediction(uid=2, iid=4, r_ui=None, est=2.9840461657069857, details={'was_impossible': False})

This prediction value is a tuple and each of the values within it can be accessed by way of indexing. Now let's put our knowledge of recommendation systems to do something interesting: making predictions for a new user!

## Obtaining User Ratings 

It's great that we have working models and everything, but wouldn't it be nice to get to recommendations specifically tailored to your preferences? That's what we'll be doing now. The first step is to create a function that allows us to pick randomly selected movies. The function should present users with a movie and ask them to rate it. If they have not seen the movie, they should be able to skip rating it. 

The function `movie_rater()` should take as parameters: 

* `movie_df`: DataFrame - a dataframe containing the movie ids, name of movie, and genres
* `num`: int - number of ratings
* `genre`: string - a specific genre from which to draw movies

The function returns:
* rating_list : list - a collection of dictionaries in the format of {'userId': int , 'movieId': int , 'rating': float}

#### This function is optional, but fun :) 

In [30]:
import random
def movie_rater(movie_df,num, genre=None):
    userRatings = []
    userID = 1000

    if genre:
        movie_df = movie_df[movie_df['genres'].str.contains(genre)]

    for i in range(num):

        movie = movie_df.sample(1)

        print(movie[['movieId', 'title', 'genres']])

        rating = input(
            'How do you rate this movie on a scale of 1-5? Press n if not seen: '
        )

        if rating.lower() == 'n':
            continue

        else:
            userRatings.append({
                'userId': userID,
                'movieId': int(movie['movieId']),
                'rating': float(rating)
            })

    return userRatings
        

In [18]:
# try out the new function here!

If you're struggling to come up with the above function, you can use this list of user ratings to complete the next segment

In [31]:
user_rating = [
    {'userId': 1000, 'movieId': 55245, 'rating': '5'},
    {'userId': 1000, 'movieId': 2491, 'rating': '4'},
    {'userId': 1000, 'movieId': 4718, 'rating': '4'},
    {'userId': 1000, 'movieId': 5990, 'rating': '3'}
]

### Making Predictions With the New Ratings
Now that you have new ratings, you can use them to make predictions for this new user. The proper way this should work is:

* add the new ratings to the original ratings DataFrame, read into a `surprise` dataset 
* train a model using the new combined DataFrame
* make predictions for the user
* order those predictions from highest rated to lowest rated
* return the top n recommendations with the text of the actual movie (rather than just the index number) 

In [32]:
## add the new ratings to the original ratings DataFrame
# Convert ratings into dataframe
new_ratings_df = pd.DataFrame(user_rating)

# Combine with original dataframe
updated_df = pd.concat([new_df, new_ratings_df], ignore_index=True)

updated_df.head()

,userId,movieId,rating
0,1,1,4
1,1,3,4
2,1,6,4
3,1,47,5
4,1,50,5


In [33]:
# train a model using the new combined DataFrame
# Reload into surprise format
reader = Reader(rating_scale=(0.5, 5))

new_data = Dataset.load_from_df(updated_df, reader)

# Build trainset
new_dataset = new_data.build_full_trainset()

# Train model
svd_2 = SVD(n_factors=50, reg_all=0.05)

svd_2.fit(new_dataset)

In [34]:
# make predictions for the user
# you'll probably want to create a list of tuples in the format (movie_id, predicted_score)
# Get movies already rated
rated_movies = list(new_ratings_df['movieId'])

# Create prediction list
predictions = []

# Predict ratings for unseen movies
for movie in df_movies['movieId'].unique():

    if movie not in rated_movies:

        pred = svd_2.predict(1000, movie)

        predictions.append((movie, pred.est))

In [35]:
# order the predictions from highest to lowest rated

ranked_movies = sorted(
    predictions,
    key=lambda x: x[1],
    reverse=True
)

ranked_movies[:10]

[(904, 4.5574283330000736),
 (1204, 4.524151640559419),
 (750, 4.50463994340752),
 (48516, 4.4949127970240745),
 (908, 4.494698502346254),
 (296, 4.489732342823677),
 (50, 4.487633560038296),
 (3435, 4.475775071677097),
 (951, 4.468078351919138),
 (1242, 4.4680596689514545)]

 For the final component of this challenge, it could be useful to create a function `recommended_movies()` that takes in the parameters:
* `user_ratings`: list - list of tuples formulated as (user_id, movie_id) (should be in order of best to worst for this individual)
* `movie_title_df`: DataFrame 
* `n`: int - number of recommended movies 

The function should use a `for` loop to print out each recommended *n* movies in order from best to worst

In [37]:
# return the top n recommendations using the 
def recommended_movies(user_ratings,movie_title_df,n):
        for i in range(n):

            movie_id = user_ratings[i][0]

            title = movie_title_df[
               movie_title_df['movieId'] == movie_id
        ]['title']

        print("Recommendation #", i+1, ":", title, '\n')
            
recommended_movies(ranked_movies,df_movies,5)

Recommendation # 5 : 690    North by Northwest (1959)
Name: title, dtype: object 



## Level Up (Optional)

* Try and chain all of the steps together into one function that asks users for ratings for a certain number of movies, then all of the above steps are performed to return the top $n$ recommendations
* Make a recommender system that only returns items that come from a specified genre

## Summary

In this lab, you got the chance to implement a collaborative filtering model as well as retrieve recommendations from that model. You also got the opportunity to add your own recommendations to the system to get new recommendations for yourself! Next, you will learn how to use Spark to make recommender systems.